In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from random import random, seed
from copy import deepcopy

In [ ]:
#funkcje grafowe z pierwszych zajec
def print_matrix(vertices, matrix):
  """
  Wypisuje na ekranie graf podany jako macierz sąsiedztwa
  """
  n = len(matrix)
  if (vertices is None) or (len(vertices) != n):
    vv = range(1, n+1)
  else:
    vv = vertices
  for i in range(n):
    print(vv[i], ':', end='')
    for j in range(n):
      if (matrix[i][j]):
        print(" ", vv[j], end="")
    print("")


def print_dict(graph):
  """
  Wypisuje na ekranie graf podany jako słownik (list) sąsiedztwa
  """
  for v in graph:
    print(v, ':', end="")
    for u in graph[v]:
      print(" ", u, end="")
    print("")


In [ ]:
#klasa graph
class Graph:
    def __init__(self, graph=None, directed=0):
        if graph is None:
            graph = {}
        self.graph = graph
        self.directed = directed

    # inicjalizator ze słownika
    @classmethod
    def from_dict(cls, graph):
        return cls(graph)

    # inicjalizator z macierzy
    @classmethod
    def from_matrix(cls, matrix, vertices = None):
        if (vertices is None) or (len(vertices) != len(matrix)):
            vertices = [*range(1, len(matrix) + 1)]
        return cls.from_dict(cls._matrix_to_dict(matrix, vertices))

    # dwie prywatne metody macierz <-> słownik
    def _matrix_to_dict(matrix, vertices: list) -> dict:
        """
        Zamienia graf podany jako macierz sąsiedztwa na słownik sąsiedztwa.
        """
        res_dict = {}
        for i, v in enumerate(vertices):
            neighbours = [vertices[j] for j, edge in enumerate(matrix[i]) if edge]
            res_dict[v] = neighbours
        return res_dict

    def _dict_to_matrix(self, _dict: dict) -> np.array:
        """
        Zamienia graf podany jako słownik sąsiedztwa na macierz sąsiedztwa.
        """
        n = len(_dict)
        vertices = [*_dict.keys()]
        matrix = np.zeros(shape = (n, n), dtype=int)
        for u,v in [
            (vertices.index(u), vertices.index(v))
            for u, row in _dict.items() for v in row
        ]:
            matrix[u][v] += 1
        return matrix

    def vertices(self) -> list:
        """
        Zwraca listę wierzchołków grafu.
        """
        return [*self.graph.keys()]

    def matrix(self) -> np.array:
        """
        Zwraca macierz sąsiedztwa grafu.
        """
        return self._dict_to_matrix(self.graph)

    # przedefiniowania sposobu wyświetlania grafów
    def __str__(self):
        res = ""
        for v in self.graph:
            res += f"{v}:"
            for u in self.graph[v]:
                res += f" {u}"
            res += "\n"
        return res

    # Poniższe dostajemy za darmo z powyższego
    def to_neighbourlist(self, filename: str):
        """
        Zapisuje graf podany jako słownik (list) sąsiedztwa do pliku (w formie listy sąsiedztwa).
        Zmienna filename zawiera pełną ścieżkę pliku
        """
        file = open(filename, "w")  # otwarcie pliku tekstowego do zapisu
        file.write(str(self))
        file.close()

    # rysowanie grafów
    def plot(self, pos=None, directed=None):
      """
      Rysuje graf używając pakietu networkx
      """
      if directed is None:
        directed = self.directed
      if directed:
        G = nx.DiGraph(self.graph)
      else:
        G = nx.Graph(self.graph)

      if pos is None:
        pos = nx.spring_layout(G)
      nx.draw(G, pos, with_labels=True)
      plt.show()

    # Modyfikacje grafów
    def add_vertex(self, vertex):
        """
        Dodaje wierzchołek do grafu
        """
        if vertex not in self.graph:
            self.graph[vertex] = []

    def del_vertex(self, vertex):
        """
        Usuwa wierzchołek z grafu
        """
        if vertex in self.graph:
            self.graph.pop(vertex)
            for u in self.graph:
                if vertex in self.graph[u]:
                    self.graph[u].remove(vertex)

    def add_arc(self, arc):
        """
        Dodaje łuk (skierowany, podany jako para wierzchołków) do grafu
        """
        u, v = arc
        self.add_vertex(u)
        self.add_vertex(v)
        if v not in self.graph[u]:
            self.graph[u].append(v)

    def add_edge(self, edge: list):
        """
        Dodaje krawędź (podaną jako para wierzchołków) do grafu
        Rozpatrujemy grafy proste, nieskierowane
        """
        u, v = edge
        if u == v:
            raise ValueError("Pętle nie są dopuszczalne!")
        self.add_vertex(u)
        self.add_vertex(v)
        if v not in self.graph[u]:
            self.graph[u].append(v)
        if u not in self.graph[v]:
            self.graph[v].append(u)

    # czytanie z plików
    @staticmethod
    def from_edges(filename: str, directed = 0):
        """
        Tworzy graf na podstawie pliku z łukami/krawędziami.
        Opis łuku/krawędzi to dwa słowa lub wierzchołka (jedno słowo).
        Nadmiarowe słowa są ignorowane.
        Zmienna filename zawiera pełną ścieżkę pliku
        """
        graph = Graph(directed=directed)
        file = open(filename, "r")          # otwarcie pliku do odczytu
        for line in file:                   # dla każdej linii w pliku
          words = line.strip().split()      # rozdziel linię na słowa
          if len(words) == 1:               # jedno słowo - opis wierzchołka
            graph.add_vertex(words[0])
          elif len(words) >= 2:             # conajmnej 2 słowa - opis krawędzi/łuku
            if directed:
              graph.add_arc([words[0], words[1]])
            else:
              graph.add_edge([words[0], words[1]])
        file.close()
        return graph

    # zapisywanie grafu do pliku w postaci listy krawędzi,
    def graph_to_edges(self, filename):
      with open(filename, 'w') as file:
        visited = set() # żeby się nie powtarzały krawędzie
        for v in self.graph:
          for u in self.graph[v]:
            if self.directed:
              file.write(f"{v} {u}\n") # każda krawędź w osobnym wierszu
            else:
              if (u, v) not in visited:
                file.write(f"{v} {u}\n")
                visited.add((v, u))

    @staticmethod
    def from_neighbourlist(filename, directed=0):
      """
      wczytywanie grafu z listy sąsiedztwa
      """
      graph = Graph(directed=directed)
      with open(filename, 'r') as file:
        for line in file:
          parts = line.strip().split(":")

          v = parts[0].strip()
          graph.add_vertex(v)

          if len(parts) > 1:
            neighbours = parts[1].strip().split()

            for u in neighbours:
              if directed:
                graph.add_arc((v,u))
              else:
                graph.add_edge((v,u))
      return graph

    @staticmethod
    def random_graph(n: int, p: float):
        """
        Tworzy losowy graf nieskierowany G(n,p)
        """
        rand_graph = Graph()
        for i in range(1, n + 1):
            rand_graph.add_vertex(i)
            for j in range(1, i):
                if random() < p:
                    rand_graph.add_edge([i, j])
        return rand_graph

    @staticmethod
    def cycle(n: int):
        """
        Tworzy graf cykliczny o n wierzchołkach
        """
        cycle = Graph()
        for i in range(n-1):
          cycle.add_edge([i+1, i+2])
        cycle.add_edge([1, n])
        return cycle


    def Prufer(self):
        """
        Zwraca kod Prüfera dla drzewa.
        Uwaga: nie jest sprawdzane, czy graf jest drzewem!!!
        kod będzie zrócony jako napis.
        """
        tr = deepcopy(self.graph)     # kopia słownika, bo go zepsuję
        code = ""
        for i in range(len(self.graph)-2):
          for x in sorted(tr):
            if len(tr[x]) == 1:     # najmniejszy liść
              break
          v = tr[x][0]    # sąsiad x
          code += f"{v} "
          tr.pop(x)
          tr[v].remove(x)
        return code.strip()


    def tree_from_Prufer(code : str):
        """
        Tworzy drzewo na podstawie kodu Prüfera.
        """
        tree = Graph()
        clist = [int(x) for x in code.strip().split()]  # kod jako lista
        n = len(clist) + 2
        vert = [x for x in range(1, n+1)]
        for v in vert:
          tree.add_vertex(v)
        for i in range(n-2):
          for x in vert:
            if x not in clist:  # najmniejszy liśc
              break
          v = clist.pop(0)    # sąsiad x
          tree.add_edge((x, v))
          vert.remove(x)
        tree.add_edge(vert)   # ostatnie 2 wierzchołki tworzą krawędź
        return tree

    def preorder(self, v, visited=None):
          if visited is None:
              visited = set()

          print(v, end=" ")
          visited.add(v)

          for u in sorted(self.graph[v]):
              if u not in visited:
                  self.preorder(u, visited)

    def postorder(self, v, visited=None):
          if visited is None:
              visited = set()

          visited.add(v)

          for u in sorted(self.graph[v]):
              if u not in visited:
                  self.postorder(u, visited)

          print(v, end=" ")


    def connected_components(self):
      """
      Zwraca listę składowych spójnych grafu, jako listę zbiorów wierzchołków.
      Uwaga: zerowy element listy zawiera wszystkie wierzchołki grafu.
      """
      def DFS(u):
        """
        Przechodzenie w głąb - funkcja wewnętrzna
        """
        for w in self.graph[u]:
          if w not in VT[0]:
            VT[0].add(w)
            VT[-1].add(w)
            DFS(w)
      """
      VT - lista zbiorów wierzchołków
      VT[0] - docelowo zbiór wszystkich wierzchołków grafu
      """
      VT = [set([])]
      for v in self.graph:
        if v not in VT[0]:    # jeżeli v nieodwiedzony
          VT[0].add(v)        # v - już odwiedzony
          VT.append(set([v])) # zalążek nowej spójnej składowej
          DFS(v)
      return VT


    def connected_components_graphs(self):
      """
      Zwraca listę spójnych składowych (nieskierowanego) grafu jako listę grafów.
      """
      components = self.connected_components()
      graphs = []
      for comp in components[1:]:
        subgraph = Graph()
        for v in comp:
          subgraph.graph[v] = self.graph[v].copy()
        graphs.append(subgraph)
      return graphs


    def distance(self, v):
      """
      Zwraca słownik odległości wierzchołka v do innych osiągalnych wierzchołków.
      Używa BFS
      """
      dist = {v:0}    # zalążek słownika
      queue = [v]     # kolejka wierzchołków
      while queue:
        u = queue.pop(0)
        for w in self.graph[u]:
          if w not in dist:
            dist[w] = dist[u] + 1
            queue.append(w)
      return dist

    #Zadanie 1.
    def ConnectedComponentsBFS(self):
      """
      Wyznacza spójne składowe grafu używając BFS.
      Zwraca listę zbiorów wierzchołków, przy czym
      VT[0] = zbiór wszystkich odwiedzonych wierzchołków.
      """

      VT = [set()]

      for v in self.graph:

          #jeśli jeszcze nieodwiedzony
          if v not in VT[0]:

              #nowa składowa
              component = set([v])

              #oznacz jako odwiedzony
              VT[0].add(v)

              #kolejka BFS (wierzcholki do odwiedzenia)
              queue = [v]

              while queue:

                  u = queue.pop(0) #dopoki cos jest w kolejce, pobieramy pierwszy

                  for w in self.graph[u]: #sprawdzamy sasiadow (z listy sasiadow wierzcholka u)

                      if w not in VT[0]:

                          VT[0].add(w)
                          component.add(w)  #dodajemy rowniez do skladowej
                          queue.append(w) #dodajemy do kolejki - potem odwiedzimy jego sasiadow

              VT.append(component)

      return VT


    #Zadanie 2.
    def TopologicalSort(self):
      """
      Wykonuje sortowanie topologiczne grafu skierowanego.
      Zwraca listę wierzchołków w porządku topologicznym.
      """

      visited = set()
      order = []

      def DFS(v):

          visited.add(v)

          for u in self.graph[v]: #odwiedzamy sasiadow, idac po wszystkich krawedziach

              if u not in visited:
                  DFS(u)

          #dodajemy po przetworzeniu sąsiadów
          order.append(v)

      for v in self.graph:

          if v not in visited:
              DFS(v)

      #odwracamy kolejność, zgodnie z zamysłem sortowania topologicznego
      order.reverse()

      return order


    #Zadanie 3.
    def StronglyConnectedComponents(self):
      """
      Wyznacza silnie spójne składowe grafu skierowanego.
      Pierwszy DFS: przechodzimy graf i zapisujemy kolejność zakończenia odwiedzania wierzchołków.
      Ta kolejność pokazuje, które wierzchołki są „źródłami” dla SCC.
      Odwracamy wszystkie krawędzie grafu (tworzymy graf transponowany).
      Drugi DFS: przetwarzamy wierzchołki w kolejności odwrotnej do zakończenia pierwszego DFS.
      Każdy uruchomiony DFS w odwróconym grafie odwiedza dokładnie jedną silnie spójną składową (SCC).
      """

      visited = set()
      order = []

      #DFS 1 (zapisujemy kolejnosc przetwarzania wierzcholkow)
      def DFS1(v):

          visited.add(v)

          for u in self.graph[v]:
              if u not in visited:
                  DFS1(u)

          order.append(v)    #dodajemy po przetworzeniu sasiadow

      for v in self.graph:   #DFS dla wszystkich wierzcholkow
          if v not in visited:
              DFS1(v)

      #odwrócenie grafu
      reversed_graph = Graph(directed=1)

      for v in self.graph:    #dodajemy wszystkie wierzcholki
          reversed_graph.add_vertex(v)

      for v in self.graph:    #odwracamy kierunki wszystkich krawedzi
          for u in self.graph[v]:
              reversed_graph.add_arc((u, v))

      #DFS 2 - szukamy SCC w odwróconym grafie
      visited.clear()

      components = []

      def DFS2(v, component):

          visited.add(v)
          component.add(v)  #dodajemy wierzcholek do aktualnej SCC

          for u in reversed_graph.graph[v]:
              if u not in visited:
                  DFS2(u, component)
      #przetwarzamy wierzcholki w odwrotnej kolejnosci przetworzenia
      while order:

          v = order.pop()

          if v not in visited:

              component = set() #nowa silnie spojna skladowa

              DFS2(v, component)

              components.append(component)

      return components




In [ ]:
#testowanie
g = Graph()

g.add_edge((1,2))
g.add_edge((2,3))

g.add_edge((4,5))
g.add_edge((5,6))

print(g.ConnectedComponentsBFS())

[{1, 2, 3, 4, 5, 6}, {1, 2, 3}, {4, 5, 6}]


In [ ]:
g = Graph(directed=1)

g.add_arc(("A", "B"))
g.add_arc(("A", "C"))
g.add_arc(("B", "D"))

print(g.TopologicalSort())

['A', 'C', 'B', 'D']


In [ ]:
g = Graph(directed=1)

g.add_arc((1,2))
g.add_arc((2,3))
g.add_arc((3,1))

g.add_arc((3,4))

print(g.StronglyConnectedComponents())

[{1, 2, 3}, {4}]
